In [5]:
import requests
from bs4 import BeautifulSoup
import csv
import os
import time

# Input/output paths
input_path = "../data/competitions.csv"
output_path = "../data/competition_logos.csv"

# Load already scraped data
existing = {}
if os.path.exists(output_path):
    with open(output_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            existing[row['competition_id']] = {
                'competition_name': row['competition_name'],
                'cup_image_url': row.get('cup_image_url', '').strip(),
                'competition_logo_url': row.get('competition_logo_url', '').strip()
            }

# Store output
result = []

# Read competitions to scrape
with open(input_path, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        comp_id = row['competition_id'].strip()
        name = row['name'].strip()
        url = row['url'].strip()

        # Use existing data if available
        entry = existing.get(comp_id, {})
        cup_done = bool(entry.get('cup_image_url'))
        logo_done = bool(entry.get('competition_logo_url'))

        if cup_done and logo_done:
            print(f"{name} ({comp_id}) — already scraped")
            result.append({
                'competition_id': comp_id,
                'competition_name': name,
                'cup_image_url': entry['cup_image_url'],
                'competition_logo_url': entry['competition_logo_url']
            })
            continue

        print(f"Scraping {name} ({comp_id})")
        cup_url = entry.get('cup_image_url', '')
        logo_url = entry.get('competition_logo_url', '')

        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            res = requests.get(url, headers=headers, timeout=30)

            if res.status_code == 200:
                soup = BeautifulSoup(res.text, 'html.parser')

                def full_url(src):
                    if src.startswith('//'):
                        return 'https:' + src
                    if src.startswith('/'):
                        return 'https://www.transfermarkt.com' + src
                    return src

                if not cup_done:
                    cup_img = soup.find('img', src=lambda s: s and "/images/erfolge/fix/" in s)
                    if cup_img:
                        cup_url = full_url(cup_img['src'])

                if not logo_done:
                    logo_img = soup.find('img', src=lambda s: s and "/images/logo/header/" in s)
                    if logo_img:
                        logo_url = full_url(logo_img['src'])
                x = "x"
                print(f"Cup: {cup_url if cup_url else 0} | Logo: {logo_url if logo_url else 0}")

            else:
                print(f"Failed to load page ({res.status_code})")

        except Exception as e:
            print(f"Error for {name}: {e}")

        result.append({
            'competition_id': comp_id,
            'competition_name': name,
            'cup_image_url': cup_url,
            'competition_logo_url': logo_url
        })

        time.sleep(1)

# Save all results
with open(output_path, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['competition_id', 'competition_name', 'cup_image_url', 'competition_logo_url']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(result)

italy-cup (CIT) — already scraped
johan-cruijff-schaal (NLSC) — already scraped
kypello-elladas (GRP) — already scraped
supertaca-candido-de-oliveira (POSU) — already scraped
russian-super-cup (RUSS) — already scraped
supercopa (SUC) — already scraped
uefa-super-cup (USC) — already scraped
Scraping superligaen (DK1)
Cup: 0 | Logo: https://tmssl.akamaized.net//images/logo/header/dk1.png?lm=1739371406
europa-league (EL) — already scraped
Scraping laliga (ES1)
Cup: 0 | Logo: https://tmssl.akamaized.net//images/logo/header/es1.png?lm=1725974302
Scraping ligue-1 (FR1)
Cup: 0 | Logo: https://tmssl.akamaized.net//images/logo/header/fr1.png?lm=1732280518
Scraping serie-a (IT1)
Cup: 0 | Logo: https://tmssl.akamaized.net//images/logo/header/it1.png?lm=1656073460
Scraping eredivisie (NL1)
Cup: 0 | Logo: https://tmssl.akamaized.net//images/logo/header/nl1.png?lm=1674743474
russian-cup (RUP) — already scraped
Scraping liga-portugal-bwin (PO1)
Cup: 0 | Logo: https://tmssl.akamaized.net//images/logo/